# TensorProgression — Companion Notebook

Concrete numerical examples for every tensor in the FIML pipeline.

**No DAFoam/OpenFOAM needed.** This notebook uses only NumPy to trace tensors
through each stage, so you can verify every computation by hand.

All dimensions match the DAFoam test case: `runRegTests_DASimpleFoamRegPar.py`

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True, linewidth=120)

## Stage 0: Problem Setup — 5-Cell Convergent Channel

In [ ]:
N_cells = 5       # number of cells
N_vars = 5        # variables per cell: Ux, Uy, Uz, p, nuTilda
N_states = N_cells * N_vars  # 25 total states
N_features = 7    # VoS, PoD, chiSA, pGradStream, PSoSS, SCurv, UOrth
N_hidden = [5, 5] # hidden layer sizes
N_output = 1      # single beta output per cell

# Illustrative converged state for a convergent channel
# Velocity increases (Bernoulli), pressure drops, nuTilda decays
cell_Ux = np.array([10.0, 10.8, 11.9, 13.2, 14.8])
cell_Uy = np.zeros(N_cells)
cell_Uz = np.zeros(N_cells)
cell_p  = np.array([48.5, 36.2, 24.3, 12.1, 0.0])
cell_nu = np.array([4.50e-5, 4.10e-5, 3.70e-5, 3.20e-5, 2.80e-5])

# Assemble the full state vector W (25 x 1)
W = np.zeros(N_states)
for i in range(N_cells):
    W[i*N_vars + 0] = cell_Ux[i]
    W[i*N_vars + 1] = cell_Uy[i]
    W[i*N_vars + 2] = cell_Uz[i]
    W[i*N_vars + 3] = cell_p[i]
    W[i*N_vars + 4] = cell_nu[i]

print(f"State vector W shape: {W.shape}")
print(f"State vector W:")
for i in range(N_cells):
    s = W[i*N_vars:(i+1)*N_vars]
    print(f"  cell {i}: Ux={s[0]:6.1f}  Uy={s[1]:4.1f}  Uz={s[2]:4.1f}  p={s[3]:6.1f}  nu={s[4]:.2e}")

## Stage 1: Jacobian ∂R/∂W — Sparsity Pattern

The FVM stencil on a 1D mesh connects each cell to its neighbours.
For the coupled system (U, p, nuTilda), this creates a **block-tridiagonal** pattern.

In [ ]:
# Build the sparsity pattern of dR/dW for the 5-cell 1D mesh.
# In 1D, cell i's residual depends on cells i-1, i, i+1.
# Each cell has 5 coupled variables.

dRdW_pattern = np.zeros((N_states, N_states), dtype=int)

for cell_i in range(N_cells):
    for cell_j in range(N_cells):
        # cell_j is in the stencil of cell_i if |i-j| <= 1
        if abs(cell_i - cell_j) <= 1:
            # All 5 variables in cell_j can affect all 5 residuals in cell_i
            for var_i in range(N_vars):
                for var_j in range(N_vars):
                    row = cell_i * N_vars + var_i
                    col = cell_j * N_vars + var_j
                    dRdW_pattern[row, col] = 1

nnz = np.sum(dRdW_pattern)
total = N_states * N_states

print(f"Jacobian dR/dW shape: {dRdW_pattern.shape}")
print(f"Non-zeros: {nnz} / {total} ({100*nnz/total:.1f}%)")
print(f"Max non-zeros per row: {dRdW_pattern.sum(axis=1).max()}")
print()
print("Sparsity pattern (. = zero, # = non-zero):")
for i in range(N_states):
    row_str = ""
    for j in range(N_states):
        row_str += "#" if dRdW_pattern[i, j] else "."
    # Add cell boundaries
    label = f"R[{i//N_vars},{['Ux','Uy','Uz',' p','nu'][i%N_vars]:>2s}]"
    print(f"{label} |{row_str}|")

## Stage 2: Graph Coloring for Efficient Jacobian Computation

For the Jacobian to be computed via finite differences efficiently,
we use **distance-2 graph coloring**. Two columns can share a color
if they don't both appear in any row.

DAFoam: `DAColoring.C:32-784`

In [ ]:
def greedy_distance2_coloring(pattern):
    """Simple greedy distance-2 coloring for a sparsity pattern."""
    n = pattern.shape[1]
    colors = -np.ones(n, dtype=int)
    
    for col in range(n):
        # Find all rows where this column has a non-zero
        rows_with_col = np.where(pattern[:, col] != 0)[0]
        
        # Find colors used by other columns that share any of these rows
        forbidden = set()
        for row in rows_with_col:
            # Find all other columns with non-zero in this row
            other_cols = np.where(pattern[row, :] != 0)[0]
            for other_col in other_cols:
                if colors[other_col] >= 0:
                    forbidden.add(colors[other_col])
        
        # Assign smallest available color
        color = 0
        while color in forbidden:
            color += 1
        colors[col] = color
    
    return colors

colors = greedy_distance2_coloring(dRdW_pattern)
n_colors = colors.max() + 1

print(f"Number of colors needed: {n_colors}")
print(f"Speedup vs naive: {N_states}/{n_colors} = {N_states/n_colors:.1f}x")
print()
print("Color assignment:")
for i in range(N_cells):
    c = colors[i*N_vars:(i+1)*N_vars]
    print(f"  cell {i}: Ux=c{c[0]:d}  Uy=c{c[1]:d}  Uz=c{c[2]:d}  p=c{c[3]:d}  nu=c{c[4]:d}")
print(f"\nInstead of {N_states} residual evaluations, we need only {n_colors}.")

## Stage 3: NN Architecture and Parameter Count

Architecture from test case: `[7] → [5] → [5] → [1]`, tanh activation.

DAFoam: `DARegression.C:652-689` (`nParameters`)

In [ ]:
# Exact parameter count matching DAFoam's nParameters() function
n_inputs = N_features   # 7
hidden = N_hidden       # [5, 5]
n_hidden_layers = len(hidden)

# Weights
n_weights = n_inputs * hidden[0]          # input → H1: 7*5 = 35
for i in range(1, n_hidden_layers):
    n_weights += hidden[i] * hidden[i-1]  # H1 → H2: 5*5 = 25
n_weights += hidden[-1] * 1               # H2 → output: 5*1 = 5

# Biases
n_biases = sum(hidden) + 1                # 5 + 5 + 1 = 11

N_params = n_weights + n_biases

print("Parameter breakdown:")
print(f"  Input({n_inputs}) → H1({hidden[0]}): {n_inputs}*{hidden[0]} = {n_inputs*hidden[0]} weights + {hidden[0]} biases")
print(f"  H1({hidden[0]}) → H2({hidden[1]}):    {hidden[0]}*{hidden[1]} = {hidden[0]*hidden[1]} weights + {hidden[1]} biases")
print(f"  H2({hidden[1]}) → Out(1):    {hidden[1]}*1 = {hidden[1]} weights + 1 bias")
print(f"  ────────────────────────────────────────")
print(f"  Total weights: {n_weights}")
print(f"  Total biases:  {n_biases}")
print(f"  Total params:  N_p = {N_params}")

## Stage 4: NN Forward Pass — Exact DAFoam Implementation

This replicates the flat-parameter forward pass from `DARegression.C:412-484`.
The weights are stored in a single flat array θ, walked with a counter.

In [ ]:
# Initialize all parameters to 0.005 (matching the test case)
theta = np.ones(N_params) * 0.005

# Illustrative feature matrix (5 cells x 7 features)
features = np.array([
    [0.48, 0.62, 0.75, -0.31, 0.44, 0.12, 0.08],  # cell 0
    [0.51, 0.58, 0.71, -0.22, 0.41, 0.15, 0.11],  # cell 1
    [0.50, 0.55, 0.67, -0.15, 0.38, 0.13, 0.09],  # cell 2
    [0.49, 0.52, 0.63, -0.08, 0.36, 0.11, 0.07],  # cell 3
    [0.47, 0.49, 0.60, -0.03, 0.33, 0.10, 0.06],  # cell 4
])

outputShift = 1.0   # β centered at 1
outputScale = 1.0


def nn_forward_dafoam(features_cell, params, hidden_sizes, activation='tanh'):
    """
    Exact replication of DARegression::compute() forward pass.
    
    Parameters are stored flat: [w1_1,1, w1_1,2, ..., b1_1, w1_2,1, ...]
    This walks the flat array with a counter, exactly like the C++ code.
    
    See: DARegression.C:412-484
    """
    n_inputs = len(features_cell)
    n_hidden = len(hidden_sizes)
    counter = 0
    
    # Allocate hidden layer values
    layer_vals = [np.zeros(s) for s in hidden_sizes]
    
    for layer_i in range(n_hidden):
        n_neurons = hidden_sizes[layer_i]
        layer_vals[layer_i][:] = 0.0
        
        for neuron_i in range(n_neurons):
            if layer_i == 0:
                # First hidden layer: input from features
                for j in range(n_inputs):
                    layer_vals[layer_i][neuron_i] += features_cell[j] * params[counter]
                    counter += 1
            else:
                # Subsequent layers: input from previous hidden layer
                for j in range(hidden_sizes[layer_i - 1]):
                    layer_vals[layer_i][neuron_i] += layer_vals[layer_i-1][j] * params[counter]
                    counter += 1
            
            # Bias
            layer_vals[layer_i][neuron_i] += params[counter]
            counter += 1
            
            # Activation
            if activation == 'tanh':
                layer_vals[layer_i][neuron_i] = np.tanh(layer_vals[layer_i][neuron_i])
            elif activation == 'sigmoid':
                layer_vals[layer_i][neuron_i] = 1.0 / (1.0 + np.exp(-layer_vals[layer_i][neuron_i]))
    
    # Output layer (no activation)
    output_val = 0.0
    for j in range(hidden_sizes[-1]):
        output_val += layer_vals[-1][j] * params[counter]
        counter += 1
    output_val += params[counter]  # output bias
    
    assert counter == len(params) - 1, f"Counter mismatch: {counter} vs {len(params)-1}"
    
    return output_val, layer_vals


# Compute beta for each cell
print("NN Forward Pass (matching DARegression.C:412-484):")
print("=" * 60)
beta = np.zeros(N_cells)

for cell_i in range(N_cells):
    raw_output, layer_vals = nn_forward_dafoam(
        features[cell_i], theta, N_hidden, 'tanh'
    )
    beta[cell_i] = outputScale * (raw_output + outputShift)
    
    print(f"\nCell {cell_i}:")
    print(f"  Input features: {features[cell_i]}")
    for li, lv in enumerate(layer_vals):
        print(f"  Hidden layer {li+1} (after tanh): {lv}")
    print(f"  Raw NN output: {raw_output:.8f}")
    print(f"  Beta = scale * (output + shift) = {outputScale} * ({raw_output:.8f} + {outputShift}) = {beta[cell_i]:.8f}")

print("\n" + "=" * 60)
print(f"Beta field: {beta}")
print(f"Max deviation from 1.0: {np.max(np.abs(beta - 1.0)):.2e}")
print("(Small because weights are small — optimizer will adjust them)")

## Stage 5: ∂R/∂β — The Sparse Diagonal Structure

Beta only enters through the SA production term: `Cb1 * S_tilde * nuTilda * beta`.

Only the nuTilda residual row depends on beta, and only on the same cell's beta.

DAFoam: `DASpalartAllmaras.C:457`

In [ ]:
# SA model constants
Cb1 = 0.1355

# S_tilde depends on vorticity and wall distance. Illustrative values:
S_tilde = np.array([120.0, 105.0, 92.0, 80.0, 70.0])  # 1/s

# Production coupling coefficient: d_i = Cb1 * S_tilde_i * nuTilda_i
d_coeff = Cb1 * S_tilde * cell_nu

# Build dR/dbeta (25 x 5)
dRdbeta = np.zeros((N_states, N_cells))
for i in range(N_cells):
    nu_row = i * N_vars + 4  # nuTilda residual is variable index 4
    dRdbeta[nu_row, i] = -d_coeff[i]  # negative because it's on the RHS

print(f"dR/dbeta shape: {dRdbeta.shape}")
print(f"Non-zeros: {np.count_nonzero(dRdbeta)} (out of {dRdbeta.size})")
print(f"\nProduction coefficients d_i = Cb1 * S_tilde * nuTilda:")
for i in range(N_cells):
    print(f"  cell {i}: d = {Cb1:.4f} * {S_tilde[i]:.1f} * {cell_nu[i]:.2e} = {d_coeff[i]:.6e}")

print(f"\ndR/dbeta matrix (showing non-zero rows only):")
for i in range(N_cells):
    row = i * N_vars + 4
    print(f"  R_nu{i} (row {row:2d}): {dRdbeta[row, :]}")

## Stage 6: The Adjoint Equation and Total Derivative

For the variance objective $J = \sum_i (U_{x,i} - U_{x,i}^{ref})^2$:

1. Compute $\partial J / \partial W$ (only Ux components are non-zero)
2. Solve $(\partial R / \partial W)^T \psi = -(\partial J / \partial W)^T$
3. Compute $dJ/d\beta = \psi^T \cdot \partial R / \partial \beta$

DAFoam: `mphys_dafoam.py:426-567`

In [ ]:
# Reference velocity (what we're trying to match)
Ux_ref = np.array([10.5, 11.2, 12.3, 13.8, 15.2])

# dJ/dW: the objective sensitivity (1 x 25)
dJdW = np.zeros(N_states)
scale = 1.0
for i in range(N_cells):
    # J = sum_i scale * (Ux_i - Ux_ref_i)^2
    # dJ/dUx_i = 2 * scale * (Ux_i - Ux_ref_i)
    dJdW[i * N_vars + 0] = 2.0 * scale * (cell_Ux[i] - Ux_ref[i])

print("Objective: J = sum_i (Ux_i - Ux_ref_i)^2")
J = scale * np.sum((cell_Ux - Ux_ref)**2)
print(f"J = {J:.6f}")
print(f"\ndJ/dW (25 x 1, showing non-zero entries):")
for i in range(N_cells):
    val = dJdW[i * N_vars + 0]
    print(f"  dJ/dUx_{i} = 2*({cell_Ux[i]:.1f} - {Ux_ref[i]:.1f}) = {val:+.4f}")
print(f"  (All other entries are zero — p, Uy, Uz, nuTilda don't enter J directly)")

In [ ]:
# Build an illustrative dR/dW Jacobian (simplified, not from actual FVM)
# This is a block-tridiagonal matrix with random-ish but physically motivated entries
np.random.seed(42)

dRdW = np.zeros((N_states, N_states))

for cell_i in range(N_cells):
    for cell_j in range(N_cells):
        if abs(cell_i - cell_j) <= 1:
            block = np.zeros((N_vars, N_vars))
            if cell_i == cell_j:
                # Diagonal block: dominant (from implicit terms)
                block = np.diag([5.0, 5.0, 5.0, 3.0, 2.0])  # diagonal dominance
                block[0, 3] = -1.0  # U depends on p (pressure gradient)
                block[3, 0] = -0.8  # p depends on U (continuity)
                block[4, 0] = -0.3  # nuTilda depends on U (through S_tilde)
            else:
                # Off-diagonal block: weaker (from face fluxes)
                sign = 1.0 if cell_j > cell_i else -1.0
                block[0, 0] = sign * 2.0   # convective flux
                block[3, 0] = sign * 0.4   # pressure correction
                block[4, 4] = sign * 0.5   # nuTilda convection
            
            ri = cell_i * N_vars
            rj = cell_j * N_vars
            dRdW[ri:ri+N_vars, rj:rj+N_vars] = block

# Solve the adjoint equation: (dR/dW)^T * psi = -(dJ/dW)^T
psi = np.linalg.solve(dRdW.T, -dJdW)

print("Adjoint equation: (dR/dW)^T * psi = -(dJ/dW)^T")
print(f"\nAdjoint vector psi ({N_states} x 1):")
for i in range(N_cells):
    p = psi[i*N_vars:(i+1)*N_vars]
    print(f"  cell {i}: psi_Ux={p[0]:+.6f}  psi_p={p[3]:+.6f}  psi_nu={p[4]:+.6f}")

# Verify: residual of adjoint equation should be near zero
residual = dRdW.T @ psi + dJdW
print(f"\nAdjoint residual ||A^T psi + dJdW||: {np.linalg.norm(residual):.2e} (should be ~0)")

In [ ]:
# Total derivative dJ/dbeta = dJ/dbeta_direct + psi^T * dR/dbeta
dJdbeta_direct = np.zeros(N_cells)  # J doesn't depend on beta directly
dJdbeta_adjoint = psi @ dRdbeta     # psi^T * dR/dbeta (25x1)^T * (25x5) = (5,)
dJdbeta = dJdbeta_direct + dJdbeta_adjoint

print("Total derivative: dJ/dbeta = dJ/dbeta_direct + psi^T * dR/dbeta")
print(f"  dJ/dbeta_direct = {dJdbeta_direct} (J doesn't depend on beta directly)")
print(f"\n  dJ/dbeta via adjoint:")
for i in range(N_cells):
    nu_row = i * N_vars + 4
    print(f"    cell {i}: psi_nu_{i} * (-d_{i}) = {psi[nu_row]:+.6f} * ({dRdbeta[nu_row, i]:+.6e}) = {dJdbeta[i]:+.6e}")

print(f"\ndJ/dbeta = {dJdbeta}")
print(f"\nInterpretation: {'Increase' if dJdbeta[0] > 0 else 'Decrease'} beta at cell 0 "
      f"to {'increase' if dJdbeta[0] > 0 else 'decrease'} J.")

## Stage 7: Forward vs Reverse Mode — The Cost Comparison

Why solving ONE adjoint equation gives gradients for ALL design variables.

In [ ]:
# Forward mode: one linear solve per design variable
# For N_p = 71 NN weights, that's 71 solves.
#
# Reverse mode: one linear solve per objective
# For 1 objective (UVar), that's 1 solve.

print("Cost comparison: Forward vs Reverse mode")
print("=" * 50)
print(f"\n  Design variables (NN weights): N_p = {N_params}")
print(f"  Objectives:                    N_obj = 1")
print(f"  Constraints:                   N_con = 2 (PVar, UProbe)")
print(f"\n  Forward mode cost: {N_params} linear solves")
print(f"  Reverse mode cost: {1 + 2} linear solves (1 obj + 2 constraints)")
print(f"\n  Speedup: {N_params}/{1+2} = {N_params/3:.1f}x")
print(f"\nFor a real optimization with 1000 NN weights:")
print(f"  Forward: 1000 solves")
print(f"  Reverse: 3 solves")
print(f"  Speedup: 333x")
print(f"\nThis is why DAFoam uses reverse-mode AD (adjoint).")

## Stage 8: dJ/dθ — Gradient Through the NN via AD

DAFoam uses CoDiPack reverse-mode AD to automatically differentiate through:
`θ → NN → β → SA equation → R(W, β(θ))`

Here we compute this manually via backpropagation for verification.

The chain: `dJ/dθ = dJ/dβ · dβ/dθ`

DAFoam: `mphys_dafoam.py:415-424` via `calcJacTVecProduct`

In [ ]:
def nn_forward_with_intermediates(features_cell, params, hidden_sizes):
    """
    Forward pass saving all intermediate values for backprop.
    Returns: output, list of (pre_activation, post_activation) for each layer.
    """
    n_inputs = len(features_cell)
    n_hidden = len(hidden_sizes)
    counter = 0
    intermediates = []  # (z_pre_activation, a_post_activation) per layer
    
    prev_layer = features_cell.copy()
    
    for layer_i in range(n_hidden):
        n_neurons = hidden_sizes[layer_i]
        n_prev = len(prev_layer)
        z = np.zeros(n_neurons)
        
        for neuron_i in range(n_neurons):
            for j in range(n_prev):
                z[neuron_i] += prev_layer[j] * params[counter]
                counter += 1
            z[neuron_i] += params[counter]  # bias
            counter += 1
        
        a = np.tanh(z)
        intermediates.append((z.copy(), a.copy(), prev_layer.copy()))
        prev_layer = a.copy()
    
    # Output layer (linear)
    output_val = 0.0
    for j in range(hidden_sizes[-1]):
        output_val += prev_layer[j] * params[counter]
        counter += 1
    output_val += params[counter]  # bias
    
    return output_val, intermediates, prev_layer


def nn_backward(dLoss_dOutput, params, hidden_sizes, intermediates, last_hidden, features_cell):
    """
    Manual backpropagation through the NN.
    Returns dLoss/dtheta (gradient w.r.t. all parameters).
    """
    n_inputs = len(features_cell)
    n_hidden = len(hidden_sizes)
    grad = np.zeros(len(params))
    
    # Reconstruct parameter layout
    # Output layer: last hidden_sizes[-1] weights + 1 bias
    # Count backwards to find output layer params
    output_start = len(params) - hidden_sizes[-1] - 1
    
    # Gradient of output layer
    delta = dLoss_dOutput  # scalar
    for j in range(hidden_sizes[-1]):
        grad[output_start + j] = delta * last_hidden[j]  # dL/dw = delta * input
    grad[output_start + hidden_sizes[-1]] = delta  # dL/dbias = delta
    
    # Propagate delta back to last hidden layer
    w_output = params[output_start:output_start + hidden_sizes[-1]]
    delta_hidden = delta * w_output  # (hidden_sizes[-1],)
    
    # Backward through hidden layers
    counter = output_start
    for layer_i in range(n_hidden - 1, -1, -1):
        z, a, prev_input = intermediates[layer_i]
        n_neurons = hidden_sizes[layer_i]
        n_prev = len(prev_input)
        
        # tanh derivative: d(tanh(z))/dz = 1 - tanh(z)^2
        dtanh = 1.0 - a**2
        delta_z = delta_hidden * dtanh  # element-wise
        
        # Find parameter start for this layer
        # Each neuron has n_prev weights + 1 bias
        layer_start = 0
        for li in range(layer_i):
            prev_size = n_inputs if li == 0 else hidden_sizes[li - 1]
            layer_start += hidden_sizes[li] * (prev_size + 1)
        
        # Compute gradients for this layer's weights and biases
        param_idx = layer_start
        delta_prev = np.zeros(n_prev)
        for neuron_i in range(n_neurons):
            for j in range(n_prev):
                grad[param_idx] = delta_z[neuron_i] * prev_input[j]
                delta_prev[j] += delta_z[neuron_i] * params[param_idx]
                param_idx += 1
            grad[param_idx] = delta_z[neuron_i]  # bias gradient
            param_idx += 1
        
        delta_hidden = delta_prev
    
    return grad


# Compute dJ/dtheta for each cell and sum
dJdtheta = np.zeros(N_params)

print("Computing dJ/dtheta = sum_cells dJ/dbeta_i * dbeta_i/dtheta")
print("=" * 60)

for cell_i in range(N_cells):
    # Forward pass with intermediates
    raw_output, intermediates, last_hidden = nn_forward_with_intermediates(
        features[cell_i], theta, N_hidden
    )
    
    # dJ/dbeta_i is what we computed earlier
    # dbeta_i/d(raw_output) = outputScale = 1.0
    # So dJ/d(raw_output) = dJ/dbeta_i * outputScale
    dLoss = dJdbeta[cell_i] * outputScale
    
    # Backprop through NN
    grad_cell = nn_backward(dLoss, theta, N_hidden, intermediates, last_hidden, features[cell_i])
    dJdtheta += grad_cell
    
    print(f"Cell {cell_i}: dJ/dbeta_{cell_i} = {dJdbeta[cell_i]:+.6e}  "
          f"|grad| = {np.linalg.norm(grad_cell):.6e}")

print(f"\nFull gradient dJ/dtheta shape: {dJdtheta.shape}")
print(f"||dJ/dtheta|| = {np.linalg.norm(dJdtheta):.6e}")

# Show first and last few entries
print(f"\nFirst 8 entries (layer 1, neuron 1 weights + bias):")
print(f"  {dJdtheta[:8]}")
print(f"Last 6 entries (output layer weights + bias):")
print(f"  {dJdtheta[-6:]}")

## Stage 9: Verify with Finite Differences

The gold standard: perturb each parameter by epsilon, recompute J, take the ratio.

In [ ]:
def compute_J_from_theta(theta_val, features, Ux_ref, cell_Ux_base, dRdW, dRdbeta_mat, N_cells, N_vars):
    """
    Simplified: compute J given NN weights theta.
    
    In reality, changing theta changes beta, which changes the SA equation,
    which changes the flow solution W*, which changes J.
    
    Here we linearize: W* ≈ W0 + (dR/dW)^{-1} * dR/dbeta * (beta - beta0)
    """
    # Compute beta from NN
    beta_new = np.zeros(N_cells)
    for i in range(N_cells):
        raw, _ = nn_forward_dafoam(features[i], theta_val, N_hidden, 'tanh')
        beta_new[i] = outputScale * (raw + outputShift)
    
    # Linearized state change: dW = -(dR/dW)^{-1} * dR/dbeta * dbeta
    dbeta = beta_new - 1.0  # deviation from baseline beta = 1
    dR = dRdbeta_mat @ dbeta
    dW = -np.linalg.solve(dRdW, dR)
    
    # New Ux values
    Ux_new = cell_Ux_base.copy()
    for i in range(N_cells):
        Ux_new[i] += dW[i * N_vars + 0]
    
    return np.sum((Ux_new - Ux_ref)**2)


# Compute FD gradient for a few parameters
eps = 1e-5
J0 = compute_J_from_theta(theta, features, Ux_ref, cell_Ux, dRdW, dRdbeta, N_cells, N_vars)

# Check first 5 and last 3 parameters
check_indices = list(range(5)) + list(range(N_params-3, N_params))

print("Finite difference verification:")
print(f"{'Index':>5s}  {'Adjoint':>12s}  {'FD':>12s}  {'Rel Error':>12s}")
print("-" * 50)

for idx in check_indices:
    theta_pert = theta.copy()
    theta_pert[idx] += eps
    J_pert = compute_J_from_theta(theta_pert, features, Ux_ref, cell_Ux, dRdW, dRdbeta, N_cells, N_vars)
    fd_grad = (J_pert - J0) / eps
    adj_grad = dJdtheta[idx]
    rel_err = abs(fd_grad - adj_grad) / (abs(fd_grad) + 1e-16)
    print(f"{idx:5d}  {adj_grad:+12.6e}  {fd_grad:+12.6e}  {rel_err:12.6e}")

print("\n(Differences arise because the linearized model is approximate.")
print(" In DAFoam, the AD gradient is exact to machine precision.)")

## Stage 10: Shape Summary — Every Tensor at a Glance

In [ ]:
print("Complete Tensor Inventory")
print("=" * 70)
tensors = [
    ("theta (NN weights)", theta, "Design variables"),
    ("features (cell 0)", features[0], "Input to NN"),
    ("beta", beta, "Correction field"),
    ("W (state vector)", W, "All flow unknowns"),
    ("dR/dW (Jacobian)", dRdW, "State Jacobian (sparse)"),
    ("dR/dbeta", dRdbeta, f"Residual-beta coupling ({np.count_nonzero(dRdbeta)} nnz)"),
    ("dJ/dW", dJdW, "Objective sensitivity to states"),
    ("psi (adjoint)", psi, "Adjoint vector"),
    ("dJ/dbeta", dJdbeta, "Gradient w.r.t. beta"),
    ("dJ/dtheta", dJdtheta, "Gradient w.r.t. NN weights"),
]

print(f"{'Tensor':<25s}  {'Shape':<12s}  {'||·||':<12s}  {'Description'}")
print("-" * 70)
for name, t, desc in tensors:
    t = np.asarray(t)
    shape_str = str(t.shape)
    norm_str = f"{np.linalg.norm(t):.4e}"
    print(f"{name:<25s}  {shape_str:<12s}  {norm_str:<12s}  {desc}")

print("\n" + "=" * 70)
print("Data flow:")
print("  theta(71) → NN → beta(5) → R(W,beta)=0 → W*(25) → J(scalar)")
print("  J(1) → dJ/dW(25) → psi(25) → dJ/dtheta(71)")